# Proteus Arc — EEG Alzheimer's Classification on OpenNeuro **ds004504**
### Conventional ML (Logistic Regression, SVM, Random Forest) with subject-independent validation

**Dataset (selected by literature review):** *A dataset of EEG recordings from Alzheimer's disease, Frontotemporal dementia and Healthy subjects* — OpenNeuro `ds004504` (Miltiadous et al., *Data* 2023, doi:10.3390/data8060095). License CC0.
88 subjects (36 AD, 23 FTD, 29 CN), 19 channels (10-20), 500 Hz, resting state eyes closed, BIDS format, raw + preprocessed (ASR + ICA) versions.

**Task:** default is **AD vs CN** (the most commonly benchmarked task on this dataset). Set `TASK` in Cell 5 for FTD vs CN, AD vs FTD, or 3-class.

**Validation:** participant-independent (`StratifiedGroupKFold`, or Leave-One-Subject-Out). Epoch-level *and* subject-level (majority vote) metrics are reported.

> ⚠️ **Run status:** this notebook has **not** been executed on the real ds004504 data by the author (the build environment had no access to OpenNeuro). The feature extraction and evaluation code was tested on synthetic data; the download/loading cell (Cell 5) is untested against the live server. Run it in Colab and check each cell's output.

## Cell 1 — Install libraries

In [ ]:
!pip install -q numpy scipy pandas scikit-learn matplotlib seaborn joblib

## Cell 2 — Import libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from scipy.signal import welch
from scipy.stats import skew, kurtosis

from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix, roc_auc_score
)

import warnings
warnings.filterwarnings("ignore")

# np.trapezoid exists in NumPy >= 2.0; fall back to np.trapz on older versions
trapezoid = getattr(np, "trapezoid", None) or np.trapz

RANDOM_STATE = 42
print("All libraries imported successfully!")

## Cell 3 — Define EEG frequency bands
Standard clinical bands used for spectral power features. Gamma is excluded.

In [ ]:
BANDS = {
    "delta": (0.5, 4.0),
    "theta": (4.0, 8.0),
    "alpha": (8.0, 13.0),
    "beta": (13.0, 30.0),
}

print("EEG frequency bands:")
for band, (low, high) in BANDS.items():
    print(f"{band.capitalize()}: {low}-{high} Hz")

## Cell 4 — EEG feature extraction
For every channel of every epoch, the following features are computed:

* **Time-domain:** mean, std, variance, RMS, min, max, range, skewness, kurtosis
* **Hjorth parameters:** activity, mobility, complexity
* **Frequency-domain (Welch PSD):** absolute and relative power in delta, theta, alpha, beta

In [ ]:
def extract_epoch_features(epoch, sfreq, channel_names=None):
    """
    Extract time-domain and frequency-domain features from one epoch.

    Parameters
    ----------
    epoch : numpy array, shape (channels, samples)
    sfreq : float, sampling frequency in Hz
    channel_names : list, optional

    Returns
    -------
    dict of extracted features
    """
    epoch = np.asarray(epoch, dtype=float)

    if epoch.ndim != 2:
        raise ValueError("Each epoch must have shape (channels, samples).")

    n_channels, n_samples = epoch.shape

    if sfreq <= 0 or n_samples < 8:
        raise ValueError("Invalid sampling frequency or insufficient samples.")

    if not np.isfinite(epoch).all():
        raise ValueError("EEG epoch contains NaN or infinite values.")

    if channel_names is None:
        channel_names = [f"ch_{i}" for i in range(n_channels)]

    if len(channel_names) != n_channels:
        raise ValueError("Channel names do not match the number of channels.")

    features = {}

    for ch_idx, ch_name in enumerate(channel_names):
        signal = epoch[ch_idx]
        prefix = str(ch_name)

        # ---- Time-domain features ----
        features[f"{prefix}_mean"] = np.mean(signal)
        features[f"{prefix}_std"] = np.std(signal)
        features[f"{prefix}_variance"] = np.var(signal)
        features[f"{prefix}_rms"] = np.sqrt(np.mean(signal ** 2))
        features[f"{prefix}_minimum"] = np.min(signal)
        features[f"{prefix}_maximum"] = np.max(signal)
        features[f"{prefix}_range"] = np.max(signal) - np.min(signal)
        features[f"{prefix}_skewness"] = skew(signal, bias=False)
        features[f"{prefix}_kurtosis"] = kurtosis(signal, bias=False)

        # ---- Hjorth parameters ----
        activity = np.var(signal)
        first_derivative = np.diff(signal)
        second_derivative = np.diff(first_derivative)
        first_var = np.var(first_derivative)
        second_var = np.var(second_derivative)

        mobility = np.sqrt(first_var / activity) if activity > 0 else 0.0
        complexity = (
            np.sqrt(second_var / first_var) / mobility
            if first_var > 0 and mobility > 0 else 0.0
        )

        features[f"{prefix}_hjorth_activity"] = activity
        features[f"{prefix}_hjorth_mobility"] = mobility
        features[f"{prefix}_hjorth_complexity"] = complexity

        # ---- Frequency-domain features (Welch PSD) ----
        freqs, psd = welch(
            signal, fs=sfreq,
            nperseg=min(n_samples, max(8, int(sfreq * 2)))
        )

        total_mask = (freqs >= 0.5) & (freqs <= min(30.0, sfreq / 2))
        total_power = (
            trapezoid(psd[total_mask], freqs[total_mask])
            if total_mask.sum() >= 2 else 0.0
        )

        for band_name, (low, high) in BANDS.items():
            effective_high = min(high, sfreq / 2)
            mask = (freqs >= low) & (freqs < effective_high)
            power = trapezoid(psd[mask], freqs[mask]) if mask.sum() >= 2 else 0.0

            features[f"{prefix}_{band_name}_power"] = power
            features[f"{prefix}_{band_name}_relative_power"] = (
                power / total_power if total_power > 0 else 0.0
            )

    return features


def build_feature_table(X_epochs, sfreq, channel_names=None):
    """X_epochs shape: (epochs, channels, samples). Returns a DataFrame."""
    rows = []
    for i, epoch in enumerate(X_epochs):
        rows.append(extract_epoch_features(epoch, sfreq, channel_names))
        if (i + 1) % 100 == 0:
            print(f"Processed {i + 1}/{len(X_epochs)} epochs")
    return pd.DataFrame(rows)


print("EEG feature extraction functions are ready.")

## Cell 5 — Download ds004504 (preprocessed derivatives) and read metadata
Uses `openneuro-py` to fetch only `participants.tsv` and the `derivatives/` folder (ASR + ICA denoised EEG, band-passed 0.5–45 Hz, re-referenced to A1-A2), so the raw data is not downloaded.

*If this download fails:* run, in a Colab cell,
`!pip install awscli` then `!aws s3 sync --no-sign-request s3://openneuro.org/ds004504 ds004504 --exclude "*" --include "participants.tsv" --include "derivatives/*"`

In [ ]:
!pip install -q openneuro-py mne

import openneuro, mne
from pathlib import Path
mne.set_log_level("ERROR")

DATA_DIR = Path("ds004504")
openneuro.download(dataset="ds004504", target_dir=str(DATA_DIR),
                   include=["participants.tsv", "derivatives"])

participants = pd.read_csv(DATA_DIR / "participants.tsv", sep="\t")
print(participants.head())
print("\nGroup counts (A=Alzheimer's, F=FTD, C=Control):")
print(participants["Group"].value_counts())

## Cell 6 — Choose the classification task and window settings
* `TASK`: `"AD_vs_CN"`, `"FTD_vs_CN"`, `"AD_vs_FTD"` or `"AD_vs_FTD_vs_CN"`
* Each recording is cut into overlapping windows (epochs). The DICE-Net paper used 30 s windows; shorter windows give more samples per subject but are more correlated.
* Because all windows from one subject share a label, **splitting must be by subject** — handled in Cell 9.

In [ ]:
TASK = "AD_vs_CN"
EPOCH_SEC = 30.0
OVERLAP = 0.5          # 50% overlap between consecutive windows

TASK_GROUPS = {
    "AD_vs_CN": ["A", "C"],
    "FTD_vs_CN": ["F", "C"],
    "AD_vs_FTD": ["A", "F"],
    "AD_vs_FTD_vs_CN": ["A", "F", "C"],
}
LABEL_NAMES = {"A": "AD", "F": "FTD", "C": "CN"}
selected = TASK_GROUPS[TASK]
sel_participants = participants[participants["Group"].isin(selected)].reset_index(drop=True)
print("Task:", TASK, "| subjects used:", len(sel_participants))
print(sel_participants["Group"].value_counts())

## Cell 7 — Load EEG and create epochs
Reads each subject's preprocessed EEGLAB `.set` file with MNE, converts to µV, and slices into windows.
Output: `X_epochs (epochs, 19, samples)`, `y`, `participant_ids`, `sfreq`, `channel_names`.

In [ ]:
def validate_eeg_data(X_epochs, y, participant_ids, sfreq):
    X_epochs = np.asarray(X_epochs); y = np.asarray(y); participant_ids = np.asarray(participant_ids)
    if X_epochs.ndim != 3: raise ValueError("EEG data must be (epochs, channels, samples).")
    if not (len(X_epochs) == len(y) == len(participant_ids)): raise ValueError("Length mismatch.")
    if not np.isfinite(X_epochs).all(): raise ValueError("NaN/inf in EEG data.")
    print("Dataset validation successful.")
    print("Epochs:", len(X_epochs), "| Channels:", X_epochs.shape[1], "| Samples/epoch:", X_epochs.shape[2])
    print("Subjects:", len(np.unique(participant_ids)), "| Classes:", np.unique(y))


def make_epochs(data, sfreq, epoch_sec=EPOCH_SEC, overlap=OVERLAP):
    """Slice (channels, samples) into windows -> (n_epochs, channels, win)."""
    win = int(epoch_sec * sfreq)
    step = max(1, int(win * (1 - overlap)))
    n = data.shape[1]
    starts = range(0, n - win + 1, step)
    return np.stack([data[:, s:s + win] for s in starts]) if len(starts) else np.empty((0, data.shape[0], win))


X_list, y_list, pid_list = [], [], []
sfreq, channel_names = None, None

for _, row in sel_participants.iterrows():
    sid = row["participant_id"]
    fpath = DATA_DIR / "derivatives" / sid / "eeg" / f"{sid}_task-eyesclosed_eeg.set"
    if not fpath.exists():
        print("Missing file, skipping:", fpath)
        continue
    raw = mne.io.read_raw_eeglab(str(fpath), preload=True)
    if sfreq is None:
        sfreq, channel_names = raw.info["sfreq"], list(raw.ch_names)
    data = raw.get_data() * 1e6           # volts -> microvolts
    ep = make_epochs(data, sfreq)
    X_list.append(ep)
    y_list += [LABEL_NAMES[row["Group"]]] * len(ep)
    pid_list += [sid] * len(ep)

X_epochs = np.concatenate(X_list)
y = np.array(y_list)
participant_ids = np.array(pid_list)
print("X_epochs:", X_epochs.shape, "| sfreq:", sfreq, "| channels:", channel_names)
validate_eeg_data(X_epochs, y, participant_ids, sfreq)

## Cell 8 — Feature extraction and class overview
Extracts 20 features per channel (time-domain, Hjorth, band powers) → 19 × 20 = **380 features** per epoch. Saved to CSV for reuse.

In [ ]:
features_df = build_feature_table(X_epochs, sfreq=sfreq, channel_names=channel_names)
print("Feature matrix:", features_df.shape)
features_df.assign(label=y, subject=participant_ids).to_csv("ds004504_features.csv", index=False)

counts = pd.DataFrame({"epochs": pd.Series(y).value_counts(),
                       "subjects": pd.Series(y).groupby(y).apply(lambda s: len(set(participant_ids[s.index])))})
display(counts)
X = features_df.to_numpy(dtype=float)

## Cell 9 — Models and participant-independent cross-validation
* **Logistic Regression**, **SVM (RBF)**, **Random Forest** as in the original pipeline (SVM `probability=False` here for speed).
* `CV_MODE = "kfold"` → `StratifiedGroupKFold` (fast, default). `CV_MODE = "loso"` → Leave-One-Subject-Out (the strictest protocol, used by DICE-Net; slower).
* Out-of-fold predictions are scored at **epoch level** and at **subject level** (majority vote across a subject's epochs).

In [ ]:
from sklearn.model_selection import LeaveOneGroupOut, cross_val_predict

CV_MODE = "kfold"     # "kfold" or "loso"
N_SPLITS = 5

models = {
    "Logistic Regression": Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE))]),
    "SVM": Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()),
        ("classifier", SVC(kernel="rbf", C=1.0, gamma="scale", class_weight="balanced", random_state=RANDOM_STATE))]),
    "Random Forest": Pipeline([("imputer", SimpleImputer(strategy="median")),
        ("classifier", RandomForestClassifier(n_estimators=300, min_samples_leaf=2, class_weight="balanced",
                                              random_state=RANDOM_STATE, n_jobs=-1))]),
}

cv = (LeaveOneGroupOut() if CV_MODE == "loso"
      else StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE))


def subject_level_vote(y_true, y_pred, groups):
    df = pd.DataFrame({"y": y_true, "p": y_pred, "g": groups})
    t = df.groupby("g")["y"].first()
    p = df.groupby("g")["p"].agg(lambda s: s.value_counts().idxmax())
    return t.values, p.reindex(t.index).values


def evaluate_cv(models, X, y, groups, cv):
    rows, preds = [], {}
    for name, model in models.items():
        print(f"Running {name} ({CV_MODE}) ...")
        yp = cross_val_predict(model, X, y, groups=groups, cv=cv, n_jobs=1)
        preds[name] = yp
        st, sp = subject_level_vote(y, yp, groups)
        labels = sorted(np.unique(y))
        row = {"Model": name,
               "Epoch Acc": accuracy_score(y, yp),
               "Epoch Bal. Acc": balanced_accuracy_score(y, yp),
               "Epoch Macro F1": f1_score(y, yp, average="macro", zero_division=0),
               "Subject Acc": accuracy_score(st, sp),
               "Subject Bal. Acc": balanced_accuracy_score(st, sp)}
        if len(labels) == 2:   # sensitivity/specificity w.r.t. the non-CN class
            pos = [l for l in labels if l != "CN"][0]
            tn_mask = (y != pos)
            row["Sensitivity"] = recall_score(y == pos, yp == pos)
            row["Specificity"] = recall_score(y != pos, yp != pos)
        rows.append(row)
    return pd.DataFrame(rows).sort_values("Epoch Macro F1", ascending=False), preds


results_df, oof_preds = evaluate_cv(models, X, y, participant_ids, cv)
display(results_df.round(4))

## Cell 10 — Confusion matrices (out-of-fold, epoch level)

In [ ]:
labels = sorted(np.unique(y))
fig, axes = plt.subplots(1, len(oof_preds), figsize=(5 * len(oof_preds), 4))
axes = np.atleast_1d(axes)
for ax, (name, yp) in zip(axes, oof_preds.items()):
    sns.heatmap(confusion_matrix(y, yp, labels=labels), annot=True, fmt="d", cmap="Blues",
                xticklabels=labels, yticklabels=labels, ax=ax, cbar=False)
    ax.set_title(name); ax.set_xlabel("Predicted"); ax.set_ylabel("True")
plt.tight_layout(); plt.show()

# Compare with published results
print("Published reference points on ds004504 (AD vs CN): DICE-Net 83.28% (LOSO); "
      "earlier classical ML study: RF ~77%, DT ~78.5% (leave-one-person-out, earlier smaller recording set).")

## Cell 11 — Model comparison plot

In [ ]:
plot_cols = [c for c in ["Epoch Acc", "Epoch Bal. Acc", "Epoch Macro F1", "Subject Acc", "Subject Bal. Acc"] if c in results_df]
ax = results_df.set_index("Model")[plot_cols].plot(kind="bar", figsize=(12, 6))
ax.set_ylim(0, 1.05); ax.set_ylabel("Score"); ax.set_title(f"ds004504 — {TASK} ({CV_MODE})")
plt.xticks(rotation=0); plt.legend(loc="lower right"); plt.tight_layout(); plt.show()

## Cell 12 — Random Forest feature importance
Fits a final Random Forest on all data **for interpretation only** (not for performance claims) and lists the top features. Literature expects theta/alpha-band changes to matter.

In [ ]:
rf = models["Random Forest"].fit(X, y)
imp = pd.DataFrame({"Feature": features_df.columns,
                    "Importance": rf.named_steps["classifier"].feature_importances_}).sort_values("Importance", ascending=False).head(20)
display(imp)
plt.figure(figsize=(10, 7)); sns.barplot(data=imp, x="Importance", y="Feature")
plt.title("Top 20 features — Random Forest"); plt.tight_layout(); plt.show()

## Cell 13 — Save the model (trained on all data; for deployment/demo, not an unbiased estimate)

In [ ]:
import joblib
joblib.dump(rf_full := models["Random Forest"], "eeg_ds004504_rf.joblib")
print("Saved:", "eeg_ds004504_rf.joblib", round(os.path.getsize("eeg_ds004504_rf.joblib")/1024, 1), "KB")
# from google.colab import files; files.download("eeg_ds004504_rf.joblib")

## Notes & limitations
* **Small sample (88 subjects):** subject-level metrics have wide confidence intervals; a single run can mislead. Repeat with different seeds / use LOSO.
* **Validation matters:** the literature (and the 2026 systematic review of ds004504) reports that epoch-level k-fold without subject grouping inflates accuracy. This notebook never mixes a subject across folds.
* **Preprocessed derivatives** were used; ASR/ICA were fitted on the whole recording by the dataset authors, which is common practice but worth noting.
* **FTD is hard to separate from AD** with resting-state EEG (multiple studies report low AD-vs-FTD accuracy).
* Research use only — not a diagnostic tool.